In [1]:
from subprocess import check_output
from parse import search
import random
from math import exp, log
from functools import lru_cache
import json

SOLVER_PATH = 'C:\\Users\\User\\Downloads\\SudokuSolver-1.2.149-win-x64\\SudokuSolverConsole.exe'

    
#@lru_cache()
def solution_count(thermos=frozenset(), cap=0):
    args = [f'-c=thermo:{idx2str(t)}' for t in thermos]
    out = json.loads(check_output([SOLVER_PATH, '--json', '-b=9', '-nt', f'-x={str(cap)}', '--hide-banner', '-c=knight', *args]))
    
    if out['type'] == 'solutionCount':
        if 0 < cap == out['count']:
            return solution_estimate(thermos)
        else:
            return out['count']
    elif out['type'] == 'error' and out['error'] == 'ERROR: The constraints are invalid (no solutions).':
        return 0
    elif out['type'] == 'error':
        raise ValueError(out['error'])

def solution_estimate(thermos=frozenset(), iterations=1000):
    args = [f'-c=thermo:{idx2str(t)}' for t in thermos]
    out = json.loads(check_output([SOLVER_PATH, '--json', '-b=9', '-t', f'-e={iterations}', '--hide-banner', '-c=knight', *args]))
    
    if out['type'] == 'estimate':
        return int(out['estimate'])
    elif out['type'] == 'error' and out['error'] == 'ERROR: The constraints are invalid (no solutions).':
        return 0
    elif out['type'] == 'error':
        raise ValueError(out['error'])

def idx2str(indices):
    return ''.join(f'R{i // 9 + 1}C{i % 9 + 1}' for i in indices)

def open_in_fpuzzles(thermos):
    args = ['-c=thermo:' + idx2str(t) for t in thermos]
    out = check_output([SOLVER_PATH, '-b=9', '-stuv', '--hide-banner', *args])
    print(out)

In [2]:
def neighbors(i):
    y,x = i // 9, i % 9
    directions = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    collect = set()
    for dy,dx in directions:
        if 0 <= y + dy < 9 and 0 <= x + dx < 9:
            collect.add(9*(y + dy) + x + dx)
    return collect

def transit(thermos):
    thermos_ = thermos.copy()
    #pick random thermo
    t = list(random.choice(thermos_))
    thermos_.remove(tuple(t))
    occupied = {i for t in thermos_ for i in t}
    
    found = False
    while not found:
        # pick random themo cell
        i = random.randint(0, len(t) - 1)
        
        if i == 0: #bulb
            options = neighbors(t[1]) - set(t) - occupied
            if options:
                t[0] = random.choice(list(options))
                found = True
        if i == len(t) - 1: #tip
            options = neighbors(t[i - 1]) - set(t) - occupied
            if options:
                t[i] = random.choice(list(options))
                found = True
        else: #middle cell
            options = neighbors(t[i - 1]).intersection(neighbors(t[i + 1])) - set(t) - occupied
            if options:
                t[i] = random.choice(list(options))
                found = True
    thermos_.append(tuple(t))
    return thermos_

In [3]:
def anneal(thermos, T=1000, T_min=0.001, alpha=0.99, repeat=True, cap=1_000_000,
           verbose=False, halt_at_unique=True, print_threshold=float('inf'),
           log_file=None, log_threshold=float('inf')):
    cur = thermos
    cur_score = log(solution_count(frozenset(cur), cap=cap))
    
    loop = True
    while loop:
        cur_T = T
        while cur_T > T_min:
            nbr = transit(cur)
            count = solution_count(frozenset(nbr), cap=cap)
            if count > 0:
                nbr_score = log(count)
                
                if halt_at_unique and nbr_score == 1:
                    return nbr
                if random.uniform(0, 1) < (p := exp(-(nbr_score - cur_score) / T)):
                    nbr_canonic = sorted(nbr, reverse=True, key=lambda x: len(x))
                    if int(exp(nbr_score)) < print_threshold:
                        print(f'accepted {nbr_canonic} with {int(exp(nbr_score)):,} solutions at T={cur_T:.4f} p={min(p, 1):.2f}')
                    if log_file and int(exp(nbr_score)) < log_threshold:
                        log_file.write(f'{int(exp(nbr_score)):02d};{nbr_canonic}\n')
                    cur = nbr
                    cur_score = nbr_score
                    cur_T *= alpha
                elif verbose:
                    print('declined', int(exp(nbr_score)), nbr)
        
        loop = repeat

In [83]:
import numpy as np
np.arange(0,81).reshape((9,9))

array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
       [ 9, 10, 11, 12, 13, 14, 15, 16, 17],
       [18, 19, 20, 21, 22, 23, 24, 25, 26],
       [27, 28, 29, 30, 31, 32, 33, 34, 35],
       [36, 37, 38, 39, 40, 41, 42, 43, 44],
       [45, 46, 47, 48, 49, 50, 51, 52, 53],
       [54, 55, 56, 57, 58, 59, 60, 61, 62],
       [63, 64, 65, 66, 67, 68, 69, 70, 71],
       [72, 73, 74, 75, 76, 77, 78, 79, 80]])

In [14]:
thermos = [(50, 42), (58, 67), (32, 24), (55, 63), (19, 20), (8, 7), (62, 52), (5, 15), (48, 38), (73, 72)]

with open('min_2CT_ak_10.txt', 'a') as f:
    anneal(thermos, T=.5, T_min=.01, alpha=0.95, log_file=f, log_threshold=10000, cap=1, verbose=False)

accepted [(50, 42), (58, 67), (32, 24), (55, 63), (8, 7), (62, 52), (5, 15), (48, 38), (73, 72), (19, 29)] with 5,968,058 solutions at T=0.5000 p=0.72
accepted [(50, 42), (58, 67), (32, 24), (55, 63), (8, 7), (62, 52), (5, 15), (48, 38), (19, 29), (73, 72)] with 6,191,246 solutions at T=0.4750 p=0.93
accepted [(50, 42), (58, 67), (32, 24), (55, 63), (8, 7), (62, 52), (48, 38), (19, 29), (73, 72), (5, 4)] with 35,022,728 solutions at T=0.4512 p=0.03
accepted [(50, 42), (58, 67), (32, 24), (55, 63), (8, 7), (62, 52), (19, 29), (73, 72), (5, 4), (48, 38)] with 25,513,658 solutions at T=0.4287 p=1.00
accepted [(50, 42), (58, 67), (32, 24), (55, 63), (8, 7), (62, 52), (19, 29), (5, 4), (48, 38), (73, 72)] with 19,986,450 solutions at T=0.4073 p=1.00
accepted [(50, 42), (58, 67), (32, 24), (8, 7), (62, 52), (19, 29), (5, 4), (48, 38), (73, 72), (55, 47)] with 3,027,497 solutions at T=0.3869 p=1.00
accepted [(50, 42), (58, 67), (32, 24), (8, 7), (62, 52), (19, 29), (48, 38), (73, 72), (55, 47

KeyboardInterrupt: 

In [133]:
(1643049 - 1264217) / 1000

378.832

In [12]:
solution_estimate([(58, 67), (32, 24), (62, 52), (5, 15), (73, 72), (50, 42), (55, 45), (19, 9), (8, 7), (48, 47)], iterations=10000)

2808548

In [5]:
solution_count(frozenset([(58, 67), (32, 24), (62, 52), (5, 15), (73, 72), (50, 42), (55, 45), (19, 9), (8, 7), (48, 47)]), cap=1_000_000)

KeyboardInterrupt: 

In [13]:
thermos = [(58, 67), (32, 24), (62, 52), (5, 15), (73, 72), (50, 42), (55, 45), (19, 9), (8, 7), (48, 47)]
open_in_fpuzzles(thermos)

b'Finding a solution with brute force:\r\n\xc9\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xcb\xcd\xcd\xcd\xcd\xcd\xbb\r\n\xba\xdb\xdb\xdb\xdb\xdb\xb3\xdb   \xdb\xb3\xdb\xdb\xdb\xdb \xba\xdb\xdb\xdb\xdb\xdb\xb3\xdb\xdb\xdb\xdb\xdb\xb3 \xdb\xdb\xdb \xba\xdb\xdb\xdb\xdb\xdb\xb3\xdb\xdb\xdb\xdb\xdb\xb3 \xdb\xdb  \xba\r\n\xba\xdb    \xb3\xdb   \xdb\xb3   \xdb \xba    \xdb\xb3\xdb   \xdb\xb3\xdb   \xdb\xba\xdb   \xdb\xb3\xdb    \xb3  \xdb  \xba\r\n\xba\xdb\xdb\xdb\xdb\xdb\xb3\xdb\xdb\xdb\xdb\xdb\xb3  \xdb  \xba \xdb\xdb\xdb\xdb\xb3 \xdb\xdb\xdb \xb3  \xdb  \xba\xdb\xdb\xdb\xdb\xdb\xb3\xdb\xdb\xdb\xdb \xb3  \xdb  \xba\r\n\xba\xdb   \xdb\xb3    \xdb\xb3 \xdb   \xba    \xdb\xb3\xdb   \xdb\xb3 \xdb   \xba    \xdb\xb3    \xdb\xb3  \xdb  \xba\r\n\xba\xdb\xdb\xdb\xdb\xdb\xb3    \xdb\xb3\xdb    \xba\xdb\xdb\xdb\xdb\xdb\xb3\xdb\xdb\xdb\xdb\xdb\xb3\xdb\xdb\xd

In [5]:
with open('min_thermos_9_8_2.txt', 'r') as f_in:
    with open('min_thermos_9_8_2.txt', 'a') as f_out:
        for line in reversed(f_in.read().splitlines()):
            anneal(eval(line.split(';')[1]), T=200, T_min=.1, alpha=0.95, cap_factor=4, max_cap=50_000, log_file=f_out, log_threshold=100, repeat=False)

accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 43), (47, 55)] with 203 solutions at T=200
accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 43), (47, 37)] with 369 solutions at T=190.0
accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 43), (47, 56)] with 134 solutions at T=180.5
accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 43), (47, 55)] with 203 solutions at T=171.475
accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 43), (64, 55)] with 99 solutions at T=162.90124999999998
accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 52), (64, 55)] with 72 solutions at T=154.75618749999998
accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 43), (64, 55)] with 99 solutions at T=147.01837812499997
accepted [(20, 29, 28, 19, 11, 3, 12, 13, 4), (68, 69, 78, 70, 62, 53, 44, 43), (64, 55)] with 99 solutions at T=139.6674592

KeyboardInterrupt: 